# Track B — MASt3R at 512 px (FINAL VERSION USED FOR SUBMISSION)

Runs MASt3R dense 3D reconstruction at the model's native 512-pixel resolution. This is the version we used for all four datasets in our final submission.

**Required runtime**: Colab with **A100 GPU** and **High-RAM** enabled. Free T4 will run out of RAM during global alignment for the 18-image Statue dataset.

**Inputs**: a folder per dataset in your Google Drive at `/MyDrive/stem_games_data/<DatasetName>/`, each containing only the image files.

**Final parameters used:**
* `size = 512` (native MASt3R resolution)
* `VOX = 0.001` (voxel size for dedup, very fine to preserve density)
* `CONF_THRESH = 1.7` for Box, Statue, Fountain
* `CONF_THRESH = 1.5` for Entrance (busier scene needs lower threshold to keep enough background points)

## 1. Verify GPU

In [ ]:
!nvidia-smi

## 2. Clone MASt3R and install dependencies

In [ ]:
import os
os.chdir('/content')
if not os.path.exists('mast3r'):
    !git clone --recursive https://github.com/naver/mast3r.git
os.chdir('/content/mast3r')
!pip install -q -r requirements.txt
!pip install -q -r dust3r/requirements.txt
!pip install -q open3d trimesh

## 3. Download MASt3R weights

In [ ]:
import os
os.makedirs('/content/mast3r/checkpoints', exist_ok=True)
!wget -nc -O /content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth https://download.europe.naverlabs.com/ComputerVision/MASt3R/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth

## 4. Mount Google Drive and load images (Option B)

Mount Drive once per session and read images directly from your shared folder — much faster than uploading via the file picker.

**Setup (one-time)**: In Google Drive, create the folder `stem_games_data/` and inside it upload the dataset subfolders (`Box`, `Entrance`, `Statue`, `Fountain`), each containing only the image files (no `.txt` files).

**Between datasets**: just change `DATASET_NAME` below and re-run this cell.

In [ ]:
from google.colab import drive
import os

# Mount Drive (no-op if already mounted)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

# >>> CHANGE THIS BETWEEN RUNS: 'Box', 'Entrance', 'Statue', or 'Fountain' <<<
DATASET_NAME = 'Statue'

DATA_DIR = f'/content/drive/MyDrive/stem_games_data/{DATASET_NAME}'

if not os.path.isdir(DATA_DIR):
    raise FileNotFoundError(
        f'Folder not found: {DATA_DIR}\n'
        f'In Drive, create stem_games_data/ and upload the dataset folders into it.'
    )

imgs = sorted([
    os.path.join(DATA_DIR, f)
    for f in os.listdir(DATA_DIR)
    if f.lower().endswith(('.png', '.jpg', '.jpeg'))
])

if not imgs:
    raise RuntimeError(f'No images found in {DATA_DIR}')

print(f'Dataset: {DATASET_NAME}')
print(f'{len(imgs)} images ready')

## 5. Run MASt3R inference and global alignment

At 512 px, inference takes ~1-2 minutes on A100; global alignment another ~1-2 minutes.

In [ ]:
import sys
sys.path.insert(0, '/content/mast3r')
sys.path.insert(0, '/content/mast3r/dust3r')

import torch
from mast3r.model import AsymmetricMASt3R
from dust3r.inference import inference
from dust3r.utils.image import load_images
from dust3r.image_pairs import make_pairs
from dust3r.cloud_opt import global_aligner, GlobalAlignerMode

device = 'cuda'
ckpt = '/content/mast3r/checkpoints/MASt3R_ViTLarge_BaseDecoder_512_catmlpdpt_metric.pth'
model = AsymmetricMASt3R.from_pretrained(ckpt).to(device)

# *** FIXED AT 512 px (model's native training resolution) ***
images = load_images(imgs, size=512)
print(f'Loaded {len(images)} images at 512 px')

pairs = make_pairs(images, scene_graph='complete', prefilter=None, symmetrize=True)
print(f'{len(pairs)} pairs')

with torch.no_grad():
    output = inference(pairs, model, device, batch_size=1, verbose=True)

scene = global_aligner(output, device=device, mode=GlobalAlignerMode.PointCloudOptimizer)
loss = scene.compute_global_alignment(init='mst', niter=300, schedule='cosine', lr=0.01)
print(f'Final alignment loss: {loss}')

## 6. Export point cloud

Apply per-dataset confidence threshold, voxel-deduplicate, and write PLY + TXT.

**Per-dataset CONF_THRESH used in our final submission:**
* Box: 1.7
* Entrance: 1.5
* Statue: 1.7
* Fountain: 1.7

In [ ]:
import numpy as np

# === SUBMISSION PARAMETERS ===
VOX = 0.001            # voxel size for dedup
CONF_THRESH = 1.7      # 1.7 for Box / Statue / Fountain; 1.5 for Entrance

pts3d  = scene.get_pts3d()    # list of (H, W, 3) tensors
imgs_t = scene.imgs           # list of (H, W, 3) numpy arrays in [0,1]
confs  = scene.get_conf()     # list of (H, W) tensors

all_pts, all_cols = [], []
for p, c, im in zip(pts3d, confs, imgs_t):
    p   = p.detach().cpu().numpy().reshape(-1, 3)
    c   = c.detach().cpu().numpy().reshape(-1)
    rgb = (np.asarray(im) * 255).astype(np.uint8).reshape(-1, 3)
    mask = c > CONF_THRESH
    all_pts.append(p[mask])
    all_cols.append(rgb[mask])

P = np.concatenate(all_pts, axis=0).astype(np.float32)
C = np.concatenate(all_cols, axis=0).astype(np.uint8)
print(f'Total points after conf filter ({CONF_THRESH}): {len(P):,}')

# Voxel dedup
coords = np.floor(P / VOX).astype(np.int64)
key = coords[:,0]*1_000_003*1_000_003 + coords[:,1]*1_000_003 + coords[:,2]
order = np.argsort(key, kind='stable')
key_s, P_s, C_s = key[order], P[order], C[order]
starts = np.concatenate([[0], np.where(np.diff(key_s) != 0)[0] + 1])
ends   = np.concatenate([starts[1:], [len(key_s)]])
P_dd = np.stack([P_s[s:e].mean(0) for s, e in zip(starts, ends)])
C_dd = np.stack([C_s[s:e].mean(0).astype(np.uint8) for s, e in zip(starts, ends)])
print(f'After voxel-dedup (VOX={VOX}): {len(P_dd):,}')

# PLY
out_ply = f'/content/{DATASET_NAME.lower()}_track_b.ply'
with open(out_ply, 'wb') as f:
    header = (f'ply\nformat binary_little_endian 1.0\nelement vertex {len(P_dd)}\n'
              f'property float x\nproperty float y\nproperty float z\n'
              f'property uchar red\nproperty uchar green\nproperty uchar blue\n'
              f'end_header\n').encode('ascii')
    f.write(header)
    dt = np.dtype([('x','<f4'),('y','<f4'),('z','<f4'),('r','u1'),('g','u1'),('b','u1')])
    arr = np.empty(len(P_dd), dtype=dt)
    arr['x'], arr['y'], arr['z'] = P_dd[:,0], P_dd[:,1], P_dd[:,2]
    arr['r'], arr['g'], arr['b'] = C_dd[:,0], C_dd[:,1], C_dd[:,2]
    f.write(arr.tobytes())
print(f'Wrote {out_ply}')

# TXT
out_txt = f'/content/{DATASET_NAME.lower()}_track_b.txt'
np.savetxt(out_txt,
           np.concatenate([P_dd, C_dd.astype(np.int32)], axis=1),
           fmt=['%.4f','%.4f','%.4f','%d','%d','%d'])
print(f'Wrote {out_txt}')

## 7. Download files

Save to `output/<dataset>_track_b.ply` and `output/<dataset>_track_b.txt` in your local project.

In [ ]:
from google.colab import files
files.download(out_ply)
files.download(out_txt)